# Data Preprocessing

This notebook prepares the startup dataset for machine learning.

The important best practice is to split the data before learning preprocessing values. The preprocessing steps are then placed inside a scikit-learn pipeline so the same steps are used during training and prediction.

## Target column: `status`

`status` is the target, also called the label. It is the value the model must learn to predict for a company.

We keep four valid classes: `operating`, `acquired`, `closed`, and `ipo`. We remove this column from `X` and keep it separately as `y` because including the answer among the input features would cause data leakage.

In [1]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    """Find the folder that contains src/data/companies.csv."""
    current_folder = Path.cwd().resolve()
    for folder in [current_folder, *current_folder.parents]:
        if (folder / "src" / "data" / "companies.csv").exists():
            return folder
    raise FileNotFoundError("Could not find src/data/companies.csv")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "src" / "data" / "companies.csv"
TARGET_COLUMN = "status"

raw_data = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw_data):,} rows and {raw_data.shape[1]} columns.")
raw_data.head()

Loaded 196,553 rows and 44 columns.


,id,Unnamed: 0.1,entity_type,entity_id,parent_id,name,normalized_name,permalink,category_code,status,...,first_milestone_at,last_milestone_at,milestones,relationships,created_by,created_at,updated_at,lat,lng,ROI
0,c:1,0,Company,1,NaN,Wetpaint,wetpaint,/company/wetpaint,web,operating,...,2010-09-05,2013-09-18,5.0,17.0,initial-importer,2007-05-25 06:51:27,2013-04-13 03:29:00,47.606209,-122.332071,15.5
1,c:10,1,Company,10,NaN,Flektor,flektor,/company/flektor,games_video,acquired,...,NaN,NaN,NaN,6.0,initial-importer,2007-05-31 21:11:51,2008-05-23 23:23:14,34.021122,-118.396467,NaN
2,c:100,2,Company,100,NaN,There,there,/company/there,games_video,acquired,...,2003-02-01,2011-09-23,4.0,12.0,initial-importer,2007-08-06 23:52:45,2013-11-04 02:09:48,37.562992,-122.325525,NaN
3,c:10000,3,Company,10000,NaN,MYWEBBO,mywebbo,/company/mywebbo,network_hosting,operating,...,NaN,NaN,NaN,NaN,NaN,2008-08-24 16:51:57,2008-09-06 14:19:18,NaN,NaN,NaN
4,c:10001,4,Company,10001,NaN,THE Movie Streamer,the movie streamer,/company/the-movie-streamer,games_video,operating,...,NaN,NaN,NaN,NaN,NaN,2008-08-24 17:10:34,2008-09-06 14:19:18,NaN,NaN,NaN


In [2]:
valid_statuses = ["operating", "acquired", "closed", "ipo"]

model_data = raw_data.copy()
model_data[TARGET_COLUMN] = model_data[TARGET_COLUMN].astype("string").str.strip().str.lower()
model_data = model_data[model_data[TARGET_COLUMN].isin(valid_statuses)].copy()
model_data = model_data.drop_duplicates().reset_index(drop=True)

print(f"Rows after target filtering and duplicate removal: {len(model_data):,}")
print(model_data[TARGET_COLUMN].value_counts())

Rows after target filtering and duplicate removal: 196,553
status
operating    183441
acquired       9394
closed         2584
ipo            1134
Name: count, dtype: Int64


In [ ]:
columns_to_drop = [
    "status",
    "id",
    "Unnamed: 0.1",
    "entity_type",
    "entity_id",
    "parent_id",
    "name",
    "normalized_name",
    "permalink",
    "domain",
    "homepage_url",
    "twitter_username",
    "logo_url",
    "logo_width",
    "logo_height",
    "short_description",
    "description",
    "overview",
    "tag_list",
    "created_by",
    "created_at",
    "updated_at",
    "first_investment_at",
    "last_investment_at",
    "first_funding_at",
    "last_funding_at",
    "first_milestone_at",
    "last_milestone_at",
    "closed_at",
    "ROI",
]

feature_columns = [column for column in model_data.columns if column not in columns_to_drop]
X = model_data[feature_columns].copy()
y = model_data[TARGET_COLUMN].copy()

print(f"Features kept for modeling: {len(feature_columns)}")
print(feature_columns)

Features kept for modeling: 16
['category_code', 'founded_at', 'closed_at', 'country_code', 'state_code', 'city', 'region', 'investment_rounds', 'invested_companies', 'funding_rounds', 'funding_total_usd', 'milestones', 'relationships', 'lat', 'lng', 'ROI']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows: {len(X_test):,}")

Training rows: 157,242
Testing rows: 39,311


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print("The preprocessor is ready and has not learned from the test data.")

Numeric features: 9
Categorical features: 7
The preprocessor is ready and has not learned from the test data.


## What we prepared

- The target is `status` with four classes.
- Identifier and free-text columns are excluded because they are not reliable general-purpose predictors.
- `closed_at` and `ROI` are excluded because they can be known only after a company outcome and could leak the answer.
- Missing numeric values will use the median.
- Missing categorical values will use the most common category.
- Categorical values will be one-hot encoded.
- Numeric values will be standardized.

The next notebook will place this preprocessor inside a model pipeline and compare four classifiers.